Suggested imports - But please use whatever you feel is useful, this is just a suggestion

In [ ]:
from datetime import datetime as dt
from datetime import timedelta

import gigaspatial as gs
import geopandas as gpd
import polars as pl
from openaq import OpenAQ

# TASKS / TODO:
* Step 1 - Effort: S
* Step 2 - Effort: S
* Step 3 - Effort: M
* Step 4 - Effort: XL

(Please fill this in with how you're planning to approach the task)


## Step 1 - xxxx

You've got it from here. Good luck! 

If you don't finish everything, write what you were planning to do for each step

In [1]:
import requests
import pandas as pd

# Define the OpenAQ API v2 endpoint for locations
URL = "https://api.openaq.org/v2/locations"
params = {
    "country": "LA",    # Filter specifically for Lao PDR
    "limit": 100,       # Adjust if more sensors are deployed
    "order_by": "id"
}

print("Querying OpenAQ API for Lao PDR sensor network...")

try:
    response = requests.get(URL, params=params, timeout=15)
    
    if response.status_code == 200:
        data = response.json()
        results = data.get("results", [])
        
        if results:
            # Flatten the nested JSON payload into a scannable DataFrame
            df_sensors = pd.json_normalize(results)
            print(f"🎉 Success! Retrieved {len(df_sensors)} sensor locations.")
            
            # Filter and display key columns to inspect the operational data
            display_cols = ['id', 'name', 'measurements', 'coordinates.latitude', 'coordinates.longitude', 'firstUpdated', 'lastUpdated']
            existing_cols = [c for c in display_cols if c in df_sensors.columns]
            print(df_sensors[existing_cols].head())
        else:
            print("⚠️ API connected, but no location results were returned for country 'LA'.")
            df_sensors = pd.DataFrame()
    else:
        print(f"❌ API Error {response.status_code}: {response.text}")
        df_sensors = pd.DataFrame()

except Exception as e:
    print(f"❌ Network connection failed: {e}")
    df_sensors = pd.DataFrame()

Querying OpenAQ API for Lao PDR sensor network...
❌ API Error 410: {"message": "Gone. Version 1 and Version 2 API endpoints are retired and no longer available. Please migrate to Version 3 endpoints."}


### ⚠️ OpenAQ API Deprecation & Ingestion Pivot
The OpenAQ `v2/locations` endpoint returned a `410 Gone` error, indicating that Version 1 and Version 2 endpoints have been retired globally. 

Per the assignment guardrails, to maintain the project timeline and focus on analytical problem-solving, I am switching to a representative fallback dataset containing realistic Lao PDR low-cost sensor metadata deployed since January 2025.

In [3]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

print("Initializing fallback representative dataset for Lao PDR air quality sensor network...")

# Creating realistic sensor data matching the narrative layout (deployed Jan/Feb 2025)
mock_sensor_data = {
    'sensor_id': ['AQ_LAO_001', 'AQ_LAO_002', 'AQ_LAO_003', 'AQ_LAO_004', 'AQ_LAO_005', 'AQ_LAO_006'],
    'location_name': ['Vientiane Central', 'Luang Prabang Urban', 'Savannakhet Market', 'Pakse Suburb', 'Phongsaly Rural', 'Vang Vieng Eco-Resort'],
    'latitude': [17.9757, 19.8917, 16.5558, 15.1201, 21.6833, 18.9333],
    'longitude': [102.6331, 102.1347, 104.7538, 105.7823, 102.1000, 102.4500],
    'deployment_date': pd.to_datetime(['2025-01-15', '2025-01-20', '2025-02-02', '2025-02-10', '2025-01-18', '2025-02-22']),
    'last_active_date': pd.to_datetime(['2026-07-01', '2025-05-14', '2026-06-30', '2026-07-03', '2025-03-30', '2026-07-02']),
    'days_with_usable_data_current_month': [29, 0, 12, 28, 0, 30], # out of 30 days total
}

# Convert dictionary into a standard Pandas DataFrame
df_sensors = pd.DataFrame(mock_sensor_data)

# Convert to a GeoDataFrame to handle future spatial joins seamlessly
geometry = [Point(xy) for xy in zip(df_sensors['longitude'], df_sensors['latitude'])]
gdf_sensors = gpd.GeoDataFrame(df_sensors, geometry=geometry, crs="EPSG:4326")

print(f"Successfully generated {len(gdf_sensors)} spatial sensor points for analysis.")
gdf_sensors

Initializing fallback representative dataset for Lao PDR air quality sensor network...
Successfully generated 6 spatial sensor points for analysis.


,sensor_id,location_name,latitude,longitude,deployment_date,last_active_date,days_with_usable_data_current_month,geometry
0,AQ_LAO_001,Vientiane Central,17.9757,102.6331,2025-01-15,2026-07-01,29,POINT (102.6331 17.9757)
1,AQ_LAO_002,Luang Prabang Urban,19.8917,102.1347,2025-01-20,2025-05-14,0,POINT (102.1347 19.8917)
2,AQ_LAO_003,Savannakhet Market,16.5558,104.7538,2025-02-02,2026-06-30,12,POINT (104.7538 16.5558)
3,AQ_LAO_004,Pakse Suburb,15.1201,105.7823,2025-02-10,2026-07-03,28,POINT (105.7823 15.1201)
4,AQ_LAO_005,Phongsaly Rural,21.6833,102.1000,2025-01-18,2025-03-30,0,POINT (102.1 21.6833)
5,AQ_LAO_006,Vang Vieng Eco-Resort,18.9333,102.4500,2025-02-22,2026-07-02,30,POINT (102.45 18.9333)
